## begin by importing all necessary packages

In [ ]:
# this is the list from demo_photometry_for_students.ipynb

import sys
import os
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pyarrow as pa
import scipy
from scipy.signal import resample_poly
import re
import json
import glob
from IPython.display import display
import h5py
import shutil

%load_ext autoreload
%autoreload 2


# this is from Adam's tidy tools
from re import search


# this is the list from Photometry data preprocessing.ipynb, from the Simpson et al. Neuron primer

from scipy.signal import find_peaks, peak_prominences
from scipy.signal import medfilt, butter, filtfilt
from scipy.stats import linregress
from scipy.optimize import curve_fit, minimize
from scipy import stats

# a few items from the code references in O'Neal et al., 2022

from scipy.sparse import csc_matrix, eye, diags
from scipy.sparse.linalg import spsolve


#set default plot properties (this also from Simpson et al., 2024)
plt.rcParams['figure.figsize'] = [10, 8] # Make default figure size larger.
plt.rcParams['axes.xmargin'] = 0          # Make default margin on x axis zero.
plt.rcParams['axes.labelsize'] = 12     #Set default axes label size 
plt.rcParams['axes.titlesize']=15
plt.rcParams['axes.titleweight']='heavy'
plt.rcParams['ytick.labelsize']= 10
plt.rcParams['xtick.labelsize']= 10
plt.rcParams['legend.fontsize']=12
plt.rcParams['legend.markerscale']=2
plt.style.use('dark_background')
    
#shouldn't need to install tdt again now that I've done it, located both in the python_tutorial_2023 and python_acw envs

# pip install tdt


# this is for the TDT example code and data processing

import tdt

%matplotlib inline


## choose the animals of interest

In [ ]:
#_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-

# define the sub-cohort

subcoh = r"D:\ACW\photometry\cohort_5_101325\extracted\IntA_SA\coh5c\processed_sessions\for_f5_fig"
#subcoh = r"D:\ACW\photometry\pilot_recordings_extracted\ACW_coh5_IT_f1_tdt"
#_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-



#_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-



# define the IDs for the individual animals


BOX2_rat = 'ACW_coh5_IT_' 
BOX3_rat = 'ACW_coh5_IT_f5'
BOX4_rat = 'ACW_coh5_IT_'
BOX5_rat = 'ACW_coh5_IT_'



subcoh_rats = {
    'BOX2_rat': str(BOX2_rat),
    'BOX3_rat': str(BOX3_rat),
    'BOX4_rat': str(BOX4_rat),
    'BOX5_rat': str(BOX5_rat)
}



#_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-

# list of the recording sessions for this sub-cohort

files = os.listdir(subcoh)

# Exclude any file that begins with "processed_"
files = [x for x in files if not x.startswith("processed_")]
files = [x for x in files if not x.startswith("multiday_processed_")]

# trim file names to blocknames
files = {x.replace('_streams_info.feather','') for x in files}
files = {x.replace('_streams_data.feather','') for x in files}
files = {x.replace('_epocs_info.feather','')   for x in files}
files = {x.replace('_epocs_data.feather','')   for x in files}
files = {x.replace('_info.feather','')         for x in files}
files = {x.replace('_streams_preproc_downsamp.feather','')         for x in files}
files = {x.replace('_streams_preproc.feather','')         for x in files}
files = {x.replace('_GCaMP_465_dFF_epocs.feather','')         for x in files}
files = {x.replace('_GCaMP_465_epocs.feather','')         for x in files}
files = {x.replace('_epoc_ts_and_indices.feather','')         for x in files}

files = {x.replace(f'_{BOX2_rat}','')         for x in files}
files = {x.replace(f'_{BOX3_rat}','')         for x in files}
files = {x.replace(f'_{BOX4_rat}','')         for x in files}
files = {x.replace(f'_{BOX5_rat}','')         for x in files}


print(f'these are the {len(files)} session IDs for rats in sub-cohort: {subcoh}')
print(files)

print()

print(f'these are the rats in this sub-cohort:')
subcoh_rats




## Doric recordings: choose the session ID and load feather files

In [ ]:
#_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-

from fp_func_acw_051926 import streams_peek_ratpack_tdt_or_doric
from fp_func_acw_051926 import remove_artifacts_dual_signal
from fp_func_acw_051926 import filt_downsamp_streams
from fp_func_acw_051926 import streams_trim_events_doric
from fp_func_acw_051926 import epoc_onsets_IntA     
from fp_func_acw_051926 import motcorr_unified
from fp_func_acw_051926 import plot_perievent_segments_fixed
from fp_func_acw_051926 import dubexp_correct
from fp_func_acw_051926 import baselinecorr
from fp_func_acw_051926 import zscore_dFF
from fp_func_acw_051926 import detect_peaks
from fp_func_acw_051926 import epoc_streams
from collections import defaultdict


#_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-
# decide if outputs should be saved 

save = False

# Save directory and info
feather_folder = r"D:\ACW\photometry\cohort_5_101325\extracted\IntA_SA\coh5c\processed_sessions\for_f5_fig\test"

#_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-


# Define the box of interest to further process
box = 'BOX3'  
    

box_to_rat = {
    'BOX2': BOX2_rat,
    'BOX3': BOX3_rat,
    'BOX4': BOX4_rat,
    'BOX5': BOX5_rat,
}

print(box_to_rat)

# Look up the rat name for this box

rat = box_to_rat[box]
print(f'The rat to be further examined is: {rat}')



#_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-_-
session_ids = sorted(files)
print(session_ids)


bad_sessions = set()

# for before, transitions
'''
bad_sessions = {
    'ACW_coh5c_coh5d_110425_0003',  # skip
    'ACW_coh5c_coh5d_110425_0006'   # skip
}
'''

segment_mode = 'during'    ####### 'before', 'during', or 'after' for either unavail preceding, avail, or unavail following. 
                            # or 'transitions'
                            ## or None if not an intermittent access file
neg_peaks = False


session_ids = [s for s in session_ids if s not in bad_sessions]

print('Sessions to be processed:')
for s in session_ids:
    print('  ', s)

print('\n--- Starting processing ---\n')

for session_id in session_ids:
    print()
    print('#-#-#-#-#--#--#--#--#--#--#---#---#---#---#---#----#----#----#----#----#--------------------------------------#')
    print()
    print(f'Processing {session_id}')
    print()

    # ---------- Load data ----------
    streams_data = pd.read_feather(
        os.path.join(subcoh, session_id + '_streams_data.feather')
    )
    streams_info = pd.read_feather(
        os.path.join(subcoh, session_id + '_streams_info.feather')
    )

    display(streams_info)
    display(streams_data)
    
    '''
    # if the nickname isn't proper
    streams_info.loc[streams_info['nickname'] == 'Headstage01', 'nickname'] = 'ACW_coh5_IT_m3'
    streams_data.loc[streams_data['nickname'] == 'Headstage01', 'nickname'] = 'ACW_coh5_IT_m3'

    # temp for fixing file 102625_0003
    streams_info.loc[streams_info['channel_name'] == 'Headstage256LockIn01', 'channel_name'] = 'Headstage01_470nm'
    streams_data.loc[streams_data['channel_name'] == 'Headstage256LockIn01', 'channel_name'] = 'Headstage01_470nm'
    streams_info.loc[streams_info['channel_name'] == 'Headstage256LockIn02', 'channel_name'] = 'Headstage01_415nm'
    streams_data.loc[streams_data['channel_name'] == 'Headstage256LockIn02', 'channel_name'] = 'Headstage01_415nm'


    display(streams_info)
    display(streams_data)

    '''

    epoc_data = pd.read_feather(
        os.path.join(subcoh, session_id + '_epocs_data.feather')
    )
    display(epoc_data)

    # ---------- Move processed files ----------
    session_id_clean = session_id.lstrip('/\\').strip()

    processed_folder = os.path.join(subcoh, "processed_sessions")
    os.makedirs(processed_folder, exist_ok=True)

    session_files = [
        f"{session_id_clean}_streams_data.feather",
        f"{session_id_clean}_streams_info.feather",
        f"{session_id_clean}_epocs_data.feather",
        f"{session_id_clean}_epocs_info.feather",
    ]

    for fname in session_files:
        src_path = os.path.join(subcoh, fname)
        if os.path.exists(src_path):
            dst_path = os.path.join(processed_folder, fname)
            shutil.move(src_path, dst_path)
            print(f"Moved {fname} → {processed_folder}")
        else:
            print(f"File not found, skipping: {fname}")
            


    rat_data, fs, ts_dict, fig_0, fig_1, data_format = streams_peek_ratpack_tdt_or_doric(
        streams_info,
        streams_data,
        box_to_rat,
        zoomtemp = [0, 5]
    )

    display(fig_0)
    display(fig_1)
    plt.close(fig_0)
    plt.close(fig_1)
    
    # Extract the signals for this rat from the rat_data dictionary
    isos_405_all = rat_data[rat]['isos_405']
    GCaMP_465_all = rat_data[rat]['GCaMP_465']
    ts_all = ts_dict[box]  # Get the corresponding time vector for that box

    
    

    GCaMP_465_all_no_arts, isos_405_all_no_arts = remove_artifacts_dual_signal(
        ts_all,
        GCaMP_465_all,
        isos_405_all,
        threshold_drop_signal1=0.10,           # amplitude drop threshold for GCaMP
        threshold_drop_signal2=0.05,           # amplitude drop threshold for isosbestic
        sudden_drop_threshold_signal1=0.02,     # sudden drop threshold for GCaMP
        sudden_drop_threshold_signal2=0.02,    # sudden drop threshold for isosbestic
        min_artifact_duration=1,                # minimum artifact length in samples
        max_artifact_duration=5000,              # max artifact length in samples
        sampling_rate=100,  
        median_filter_size=1000,
        baseline_exclusion_seconds=0,# <-- ignore first seconds for artifact detection
        manual_windows=None, #[(2135, 2150)], #[(3225,3235)],    # list of tuples
        plot=True,
        signal1_label='GCaMP_465',
        signal2_label='isos_405',
        zoom_start = 390,
        zoom_window = 10
    )


    plt.show()
    plt.close()
    
    new_fs = fs/5  # sampling rate of 20 Hz for Doric data, which saves the demodulated signals at 100 Hz.

    filt = 8

    print(new_fs)


    isos_405_filt_downsamp, GCaMP_465_filt_downsamp, ts_downsamp, fig_8, fig_zoom = filt_downsamp_streams(
        isos_405_all_no_arts, 
        GCaMP_465_all_no_arts, 
        ts_all, fs, 
        new_fs,
        filt,
        zoom_start=300, 
        zoom_window=10
    )

    display(fig_8)
    display(fig_zoom)
    plt.close(fig_8)
    plt.close(fig_zoom)
    
    print("isos_405 length:", isos_405_all.size)
    print("isos_405_downsamp length:", isos_405_filt_downsamp.size)
    print("GCaMP_465 length:", GCaMP_465_all.size)
    print("GCaMP_465_downsamp length:", GCaMP_465_filt_downsamp.size)
    print("ts_all length:", ts_all.size)
    print("ts_downsamp length:", ts_downsamp.size)

    print()
    print(f'the duration of the stream data is {ts_downsamp[-1]} seconds, aka {ts_downsamp[-1]/60} mins')

    display(epoc_data[epoc_data['input code'].str.contains(box, case=False, na=False)])
    print('detected data format: Doric')
    
    
    # Analysis parameters


    default_dropsec = 10  # seconds

    startcode = 0     # This is 0 except in cases like IntA where there are multiple possible starts

    #dropsec_code = 'D09' # for the new Med PC programs, beginning 10.01.25

    if box == 'BOX2':
        dropsec_code = 'D09'
        
    elif box == 'BOX3':
        dropsec_code = 'D13'
            
    elif box == 'BOX4':
        dropsec_code = 'D25'
            
    elif box == 'BOX5':
        dropsec_code = 'D29'

    
    #~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~#
    
    keepmin_streams = 15 # mins
    
            
    if segment_mode == 'during':
        pre_buffer_sec = 300 #0  #seconds
        post_buffer_sec = 15 # seconds
        tzoom=[-10,30]
        
    elif segment_mode == None:
        pre_buffer_sec = 10 # seconds
        post_buffer_sec = 15 # seconds
        tzoom=[-10,60]
        
    elif segment_mode == 'transitions':
        pre_buffer_sec = 0 # seconds
        post_buffer_sec = 0 # seconds
        tzoom=[50,75]
        
    else:
        pre_buffer_sec = 0 # seconds
        post_buffer_sec = 0 # seconds
        tzoom=[-10,15]
    
    
    
    # Try to load epoc data and determine dropsec_streams

    epoc_data = None
    dropsec_streams = default_dropsec

    try:
        #epoc_data_path = os.path.join(subcoh + session_id + '_epocs_data.feather')  # already loaded above
        #epoc_data = pd.read_feather(epoc_data_path)
        # Path to the moved file
        epoc_data_path = os.path.join(subcoh, "processed_sessions", f"{session_id}_epocs_data.feather")

        # Load epoc data if it exists
        if os.path.exists(epoc_data_path):
            epoc_data = pd.read_feather(epoc_data_path)
            #display(epoc_data)
        else:
            print(f"Epoc data file not found in processed_sessions: {epoc_data_path}")
            epoc_data = None


        # 🔍 DEBUG: Show unique TTL codes for the current box
        # Extract the D## part from the 'input code' column
        epoc_data['D_code'] = epoc_data['input code'].str.extract(r'^(D\d{2})')

        # Get unique D## codes
        unique_d_codes = epoc_data['D_code'].dropna().unique()

        # Display the result
        print(f"\n📋 Unique D## codes found: {unique_d_codes}")
        
        # Extract onsets for the selected TTL code
        # Define the allowed D## codes
        allowed_d_codes = ['D09', 'D13', 'D25', 'D29']

        # Filter the DataFrame
        filtered_rows = epoc_data[
            epoc_data['input code'].str.contains(box, case=False, na=False) &  # case-insensitive match to box
            epoc_data['input code'].str.extract(r'^(D\d{2})')[0].isin(allowed_d_codes)  # extract D## and filter
        ]

        # Extract onset values
        dropsec_all = filtered_rows['onset'].values
        
        
        # 🧾 Interpret the TTL code
        if dropsec_code in ['D09', 'D13', 'D25', 'D29']:
            print('📌 dropsec_code reflects beginning of the Med Associates program')

        if dropsec_code in ['D09', 'D13', 'D25', 'D29'] and epoc_data['input code'].str.contains('Avail').any():
            print('📌 dropsec_code reflects onset of drug availability')
        

        print(f'\n⏱ dropsec code {dropsec_code} corresponds to timepoints: {dropsec_all}\n')

        # Select the event instance to use
        if len(dropsec_all) > startcode:
            dropsec_streams = dropsec_all[startcode]
            print(f'✅ Using dropsec = {dropsec_streams:.3f} seconds from epoc data.')
        else:
            dropsec_streams = default_dropsec
            pre_buffer_sec = 0 # seconds
            post_buffer_sec = 0 # seconds
            print(f'⚠️ No TTL onset found at index {startcode}. Using default dropsec = {default_dropsec} seconds, no buffers')

            
    except FileNotFoundError as e:
        dropsec_streams = default_dropsec
        epoc_data = None
        print(f'\n⚠️ Missing file "{e.filename}". Using default dropsec = {default_dropsec} seconds.')

    except Exception as e:
        dropsec_streams = default_dropsec
        epoc_data = None
        print(f'\n⚠️ Unexpected error: {e}. Using default dropsec = {default_dropsec} seconds.')



    #print(f'\nThe time segment preceding the trial begin is {dropsec_streams} seconds.')

    if 'noncont' in subcoh:
        noncont = True
    else:
        noncont = False



    # Bounds for final plot
    x = [0, 10]
    y1 = [(0.9 * np.mean(GCaMP_465_filt_downsamp)), (1.1 * np.mean(GCaMP_465_filt_downsamp))]
    y2 = [(0.9 * np.mean(isos_405_filt_downsamp)), (1.1 * np.mean(isos_405_filt_downsamp))]




    # Run stream trimming
    
    fig_2a, fig_2b, isos_405, GCaMP_465, ts = streams_trim_events_doric(
        isos_405_filt_downsamp,
        GCaMP_465_filt_downsamp,
        dropsec_streams,
        keepmin_streams,
        ts_downsamp,
        new_fs,
        y1,
        y2,
        x,
        startcode,
        epoc_data=epoc_data,  # can now safely be None
        box=box,
        noncont=False,
        tzoom=tzoom,
        segment_mode = segment_mode,
        pre_buffer_sec=pre_buffer_sec, 
        post_buffer_sec=post_buffer_sec
    )

    display(fig_2a)
    display(fig_2b)
    plt.close(fig_2a)
    plt.close(fig_2b)
    
    epoc_data_path = os.path.join(subcoh, "processed_sessions", f"{session_id}_epocs_data.feather")

    # Load epoc data if it exists
    if os.path.exists(epoc_data_path):
        epoc_data = pd.read_feather(epoc_data_path)
        #display(epoc_data)
    else:
        print(f"Epoc data file not found in processed_sessions: {epoc_data_path}")
        epoc_data = None

    
    ####  importantly, these codes should match those above for the stream trimming!

    keepmin_epocs = keepmin_streams   # what is the duration in minutes of the epoc data I want to examine? 
    #keepmin_epocs = 90 # if I want to see all events


    print(f'examining {keepmin_epocs} minutes of the epoc data')

    dropsec_code = dropsec_code   # inherited from above. this is 1 for most cases, beginning of med pc program, but is 3 for start of IntA and 2 for remaining available periods

    print(f'the dropsec code is {dropsec_code}')

    startcode = startcode    # again, inherited from above. which index of dropsec_code to use, if there are multiple options. E.g. 1 would be the index for the second drug available onset. 

    print(f'the start code is {startcode}')

    #### 

    epoc_ts_and_indices, ts_epocs = epoc_onsets_IntA(epoc_data, keepmin_epocs, ts, box, new_fs, dropsec_streams, segment_mode, data_format)
    print('Intermittent access self-admin epoc_ts_and_indices:')
    epoc_ts_and_indices.columns.name = f'{rat} epoc timestamps'
    display(epoc_ts_and_indices)
    
    
    
    #~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~

    ## motion correction using a linear fit between isosbestic and GCaMP streams


    # Time vector and zoom window
    #tzoom = (500,750)  # example limits in seconds
    tzoom = (-10,300)


    # ---  Linear regression or polyfit motion correction ---
    GCaMP_465_motcorr, est_motion, slope_linreg, intercept_linreg, r2_linreg, figs_linreg = motcorr_unified(
        ts=ts,
        isos=isos_405,
        gcamp=GCaMP_465,
        tzoom=tzoom,
        method="linregress",  #'linregress' or 'polyfit' or 'sliding_window' or 'freq_domain', though the latter two not working yet
        clamp_negative=True,
        r2_threshold=0.0001,
        plot=True
    )

    for fig in figs_linreg:
        display(fig)
        plt.close(fig)

    if segment_mode == 'transitions':
        event_filter='drug available'

    elif segment_mode == 'during':
        event_filter='infusion'

    elif segment_mode == None:
        event_filter='infusion'

    elif segment_mode in ['before', 'after']:
        event_filter='active lever'


    plot_perievent_segments_fixed(
        epoc_ts_and_indices=epoc_ts_and_indices,    # your DataFrame with event timestamps
        new_fs=new_fs,                                  # your sampling frequency
        ts=ts,                                      # your full time vector
        GCaMP_465=GCaMP_465,                        # raw GCaMP signal
        isos_405_nocuearts=isos_405,     # isosbestic signal without cue artifacts
        est_motion=est_motion,                      # estimated motion signal
        GCaMP_465_motcorr=GCaMP_465_motcorr,      # motion-corrected GCaMP signal
        event_filter=event_filter,
        pre_window_sec=4.0,
        post_window_sec=8.0
    )

    #~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~


    #~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~

    ## Correct photobleaching for isosbestic and 465 channels by subtracting a double exponential fit 

   

    tzoom = [0,ts[-1]] 

    isos_405_expfit, isos_405_expcorr, GCaMP_465_expfit, GCaMP_465_motcorr_expcorr, fig_9, fig_10 = dubexp_correct(ts, isos_405, GCaMP_465_motcorr, tzoom, downsample_factor = 10)

    display(fig_9)
    display(fig_10)
    plt.close(fig_9)
    plt.close(fig_10)

    #~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~


    #~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~

    ## Correct for photobleaching, baseline drift using an adaptive iteratively reweighted penalized least squares algorithm (airPLS)

 

    lambda_ = 10000  #e.g. 100000 for longer, 10000 for shorter recordings
    porder = 1   # e.g. 1
    itermax = 50  # e.g. 50
    tzoom = [0,ts[-1]] 

    isos_405_baseline, GCaMP_465_baseline, isos_405_expcorr_airPLS, GCaMP_465_motcorr_expcorr_airPLS, fig_a, fig_c = baselinecorr(ts, isos_405, isos_405_expcorr, GCaMP_465, GCaMP_465_motcorr_expcorr, tzoom, lambda_, porder, itermax)

    display(fig_a)
    display(fig_c)
    plt.close(fig_a)
    plt.close(fig_c)

    #~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~


    #~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~

    ### normalization by Z-scoring (subtract the mean, divide by the standard deviation) and dF/F (subtract the motion correction, then divide by the estimated motion from above)
    # these normalization methods from Simpson et al., 2024

    #tzoom = [ts[-1]/20,ts[-1]/16]
    #tzoom = [0,60]
    tzoom = [0,ts[-1]]
    yzoom = [-10,10]

    isos_405_expcorr_airPLS_zscore, GCaMP_465_motcorr_expcorr_airPLS_zscore, GCaMP_465_motcorr_expcorr_airPLS_dFF, fig_a, fig_b = zscore_dFF(ts, isos_405_expcorr_airPLS, GCaMP_465_motcorr_expcorr_airPLS, est_motion if est_motion is not None else GCaMP_465_baseline, GCaMP_465_baseline, tzoom, yzoom)

    display(fig_a)
    display(fig_b)
    plt.close(fig_a)
    plt.close(fig_b)
    
    #~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~


    #~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~

    ## concatenate the streams of interest into one dataframe

    streams_preproc = pd.DataFrame({
        'ts': ts,
        'isosbestic 405nm': isos_405,
        'isosbestic 405nm, artifacts corrected (N/A for unavail, transitions)': isos_405,
        'GCaMP 465nm': GCaMP_465,

        'estimated motion': est_motion,
        'GCaMP 465nm motion correction': GCaMP_465_motcorr,
        'GCaMP 465nm motion correction, exponential fit': GCaMP_465_motcorr_expcorr,
        'GCaMP 465nm motion correction, exponential fit, airPLS': GCaMP_465_motcorr_expcorr_airPLS,
        'GCaMP 465nm motion correction, exponential fit, airPLS, z-scored': GCaMP_465_motcorr_expcorr_airPLS_zscore,
        'GCaMP 465nm motion correction, exponential fit, airPLS, dF/F': GCaMP_465_motcorr_expcorr_airPLS_dFF,

        'isosbestic 405nm exponential fit': isos_405_expcorr,
        'isosbestic 405nm exponential fit, airPLS': isos_405_expcorr_airPLS,
        'isosbestic 405nm exponential fit, airPLS, z-scored': isos_405_expcorr_airPLS_zscore,

    })

    streams_preproc.columns.name = f'{rat} preprocessed streams'
    display(streams_preproc)

    #~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~

    # peak detection
    if neg_peaks:
        data_of_interest_peaks = {
            'name': 'GCaMP_465_motcorr_expcorr_airPLS_zscore',
            'data': GCaMP_465_motcorr_expcorr_airPLS_zscore
        }



        tzoom = ([100,200])

        mph=1.96    # minimum peak height, in z-score or dFF depending on data 
                     # z-score 1.96 for two tailed 95% confidence, 2.58 for a 99% confidence level, 
        mpd=100
        threshold=0
        prominence=1.5    
        auc_peak_window = 3  # this window depends on how clear and large the peaks are... e.g. 2 or 3 seconds wouldn't be sufficient for ACW_25 
        peak_trange=[-5, 10]          # ← Extracts peri-peak segments: 5s before to 5s after
        baseline_trange=[-5, 2]       # ← Baseline correction using 3s window starting 5s before peak



        peaks, peak_freq, peak_amps, aucs, raw_peak_df, baselined_peak_df = detect_peaks(
            ts=ts,
            new_fs=new_fs,
            x=data_of_interest_peaks['data']*-1,  # invert to detect dips
            mph=mph,
            mpd=mpd,
            threshold=threshold,
            prominence=prominence,
            edge='rising',
            kpsh=False,
            valley=False,
            show=True,
            ax=None,
            title=True,
            tzoom=tzoom,
            plot_histogram=True,
            plot_overlay=True,
            overlay_window=8,
            auc_peak_window=auc_peak_window,
            peak_trange=peak_trange,            
            baseline_trange=baseline_trange,
            pre_buffer_sec=pre_buffer_sec,           # seconds to exclude at start, if buffer time was included in the segment extraction above
            post_buffer_sec=post_buffer_sec,
            max_slope=25,   # 15 for GCaMP6s, 25 for GCamP8s          # dF/F per second (e.g., 10)
            min_width_sec=0.1,         # e.g., 0.05
            max_width_sec=None,         # optional
            max_amp_z=8,              # robust z-score (e.g., 6)
            show_rejected=True,
            plot_rejected_overlay=True
        )




        print()
        print(f'peak frequency is: {peak_freq} Hz')
        print()
        print('peak amplitudes:')
        print(peak_amps)
        print()
        print(f'mean and SEM of peak amplitudes: {np.mean(peak_amps)} +- {np.std(peak_amps, ddof=1) / np.sqrt(len(peak_amps))}')


        # create a dataframe that contains the mean and sem for peak_amp, as well as peak_freq and the mph used in the detection

        peak_amps = pd.DataFrame({"peak amplitudes": peak_amps})*-1
        peak_aucs = pd.DataFrame({"peak AUCs": aucs})*-1

        # Create dictionary with peak parameter info
        peak_parameters = {
            "data stream analyzed": data_of_interest_peaks['name'],
            "peak detection threshold": mph,
            "minimum peak distance": mpd,
            "threshold": threshold,
            "prominence": prominence,
            "peri-peak window": peak_trange,
            "baseline correction interval": baseline_trange,
            "AUC window (sec)": auc_peak_window,
            'duration of time segment (sec)': ts[-1] - post_buffer_sec,
            'number of peaks detected': len(peak_amps),
            "peak frequency (Hz)": peak_freq,
            "peak amplitude mean": float(np.mean(peak_amps))*-1,
            "peak amplitude SEM": float(np.std(peak_amps, ddof=1) / np.sqrt(len(peak_amps))),
            "peak AUC mean": float(np.mean(peak_aucs))*-1,
            "peak AUC SEM": float(np.std(peak_aucs, ddof=1) / np.sqrt(len(peak_aucs)))
        }

        # Convert to one-column DataFrame and name the column with `rat`
        peak_df = pd.DataFrame.from_dict(peak_parameters, orient='index')
        peak_df = peak_df.reset_index() 
        peak_df.columns = ['peak measurements', rat]  # Set column header to the string variable `rat`
        peak_df = peak_df.astype(str)


        peak_df
    
    else:
        
        data_of_interest_peaks = {
            'name': 'GCaMP_465_motcorr_expcorr_airPLS_zscore',
            'data': GCaMP_465_motcorr_expcorr_airPLS_zscore
        }



        tzoom = ([100,200])

        mph=1.96    # minimum peak height, in z-score or dFF depending on data 
                     # z-score 1.96 for two tailed 95% confidence, 2.58 for a 99% confidence level, 
        mpd=100
        threshold=0
        prominence=1.5    
        auc_peak_window = 3  # this window depends on how clear and large the peaks are... e.g. 2 or 3 seconds wouldn't be sufficient for ACW_25 
        peak_trange=[-5, 10]          # ← Extracts peri-peak segments: 5s before to 5s after
        baseline_trange=[-5, 2]       # ← Baseline correction using 3s window starting 5s before peak



        peaks, peak_freq, peak_amps, aucs, raw_peak_df, baselined_peak_df = detect_peaks(
            ts=ts,
            new_fs=new_fs,
            x=data_of_interest_peaks['data'],  # invert to detect dips
            mph=mph,
            mpd=mpd,
            threshold=threshold,
            prominence=prominence,
            edge='rising',
            kpsh=False,
            valley=False,
            show=True,
            ax=None,
            title=True,
            tzoom=tzoom,
            plot_histogram=True,
            plot_overlay=True,
            overlay_window=8,
            auc_peak_window=auc_peak_window,
            peak_trange=peak_trange,            
            baseline_trange=baseline_trange,
            pre_buffer_sec=pre_buffer_sec,           # seconds to exclude at start, if buffer time was included in the segment extraction above
            post_buffer_sec=post_buffer_sec,
            max_slope=25,   # 15 for GCaMP6s, 25 for GCamP8s          # dF/F per second (e.g., 10)
            min_width_sec=0.1,         # e.g., 0.05
            max_width_sec=None,         # optional
            max_amp_z=8,              # robust z-score (e.g., 6)
            show_rejected=True,
            plot_rejected_overlay=True
        )




        print()
        print(f'peak frequency is: {peak_freq} Hz')
        print()
        print('peak amplitudes:')
        print(peak_amps)
        print()
        print(f'mean and SEM of peak amplitudes: {np.mean(peak_amps)} +- {np.std(peak_amps, ddof=1) / np.sqrt(len(peak_amps))}')


        # create a dataframe that contains the mean and sem for peak_amp, as well as peak_freq and the mph used in the detection

        peak_amps = pd.DataFrame({"peak amplitudes": peak_amps})
        peak_aucs = pd.DataFrame({"peak AUCs": aucs})

        # Create dictionary with peak parameter info
        peak_parameters = {
            "data stream analyzed": data_of_interest_peaks['name'],
            "peak detection threshold": mph,
            "minimum peak distance": mpd,
            "threshold": threshold,
            "prominence": prominence,
            "peri-peak window": peak_trange,
            "baseline correction interval": baseline_trange,
            "AUC window (sec)": auc_peak_window,
            'duration of time segment (sec)': ts[-1] - post_buffer_sec,
            'number of peaks detected': len(peak_amps),
            "peak frequency (Hz)": peak_freq,
            "peak amplitude mean": float(np.mean(peak_amps)),
            "peak amplitude SEM": float(np.std(peak_amps, ddof=1) / np.sqrt(len(peak_amps))),
            "peak AUC mean": float(np.mean(peak_aucs)),
            "peak AUC SEM": float(np.std(peak_aucs, ddof=1) / np.sqrt(len(peak_aucs)))
        }

        # Convert to one-column DataFrame and name the column with `rat`
        peak_df = pd.DataFrame.from_dict(peak_parameters, orient='index')
        peak_df = peak_df.reset_index() 
        peak_df.columns = ['peak measurements', rat]  # Set column header to the string variable `rat`
        peak_df = peak_df.astype(str)


        peak_df

    #~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
    ## perievent analysis ##
    
    # ----------------------------
    # Define data streams
    # ----------------------------
    data_streams = {
        "GCaMP": {
            "name": "GCaMP 465nm motion correction, exponential fit, airPLS, z-scored",
            "data": streams_preproc["GCaMP 465nm motion correction, exponential fit, airPLS, z-scored"]
        },
        "isosbestic": {
            "name": "isosbestic 405nm exponential fit, airPLS, z-scored",
            "data": streams_preproc["isosbestic 405nm exponential fit, airPLS, z-scored"]
        }
    }

    # ----------------------------
    # Event lists by segment mode
    # ----------------------------
    if segment_mode == "during":
        event_list = ["drug infusion", "active lever", "inactive lever"]
    elif segment_mode == None:
        event_list = ["program start", "missed infusion", "active lever", "inactive lever"]  # missed or drug infusion depending on training / PR versus extinction / CR
    elif segment_mode in ("before", "after"):
        event_list = ["active lever", "inactive lever"]
    elif segment_mode == "transitions":
        event_list = ["drug available", "available_unavailable"]
    else:
        raise ValueError(f"Unknown segment_mode: {segment_mode}")

    if noncont:
        event_list = ["program start", "cue light", "houselight"]
        
    # ----------------------------
    # Analysis parameters
    # ----------------------------
    trange = [-10, 25]
    baseline_trange = [-5, -4]
    tzoom = [0, ts[-1]]

    cue_duration = 4.0
    subset = None

    approach_window = (-3, 0)
    auc_pre_window = (-3, 0)
    cue_window = (0, 4)
    auc_post_window = (0, 4)

    # ----------------------------
    # Storage container
    # ----------------------------
    perievent_results = defaultdict(dict)

    # ============================
    # MAIN LOOP
    # ============================
    for data_key, data_info in data_streams.items():
        print(f"\n🧪 Running perievent analysis for data type: {data_key}")
        stream = data_info["data"]

        # ----------------------------
        # Count real events first
        # ----------------------------
        event_counts = {
            event: (epoc_ts_and_indices["event"] == event).sum()
            for event in event_list
        }
        max_n_events = max(event_counts.values()) if event_counts else 0


        # ============================
        # REAL EVENTS
        # ============================
        for event in event_list:
            print(f"  ▶ Event: {event} ({event_counts[event]} events)")

            label = f"{rat} | {data_key} | {event} | {segment_mode}"

            outputs = epoc_streams(
                epoc_ts_and_indices,
                trange,
                stream,
                new_fs,
                ts,
                event,
                tzoom,
                cue_duration,
                subset=subset,
                cue_window=cue_window,
                auc_pre_window=auc_pre_window,
                auc_post_window=auc_post_window,
                approach_window=approach_window,
                plot_auc_region=True,
                baseline_trange=baseline_trange,
                subset_plots=None,
                pre_buffer_sec=pre_buffer_sec,
                label=label,
            )

            (
                epocs,
                epocs_baselined,
                epocs_mean_std_sem,
                mean_epoc_stream,
                std_epoc_stream,
                sem_epoc_stream,
                fig_5,
                fig_15,
                (cue_mean, cue_sem),
                (auc_pre_mean, auc_pre_sem),
                (auc_post_mean, auc_post_sem),
                (approach_mean, approach_sem),
                subset_figs, 
            ) = outputs

            label_base = f"{rat} | {data_key} | {event}"
            epocs.columns.name = f"{label_base} streams"
            epocs_mean_std_sem.columns.name = f"{label_base} mean, std, sem"

            # ----------------------------
            # Metadata dictionary
            # ----------------------------
            event_parameters = {
                "rat": rat,
                "data type": data_key,
                "preprocessed data stream": data_info["name"],
                "event type examined": event,
                "event count": epocs.shape[1],
                "trange": trange,
                "baseline_trange": baseline_trange,
                "subset": subset,

                "approach window": str(approach_window),
                "mean approach response": f"{approach_mean:.3f}",
                "SEM approach response": f"{approach_sem:.3f}",

                "cue window": str(cue_window),
                "mean cue response": f"{cue_mean:.3f}",
                "SEM cue response": f"{cue_sem:.3f}",

                "pre-event AUC window": str(auc_pre_window),
                "mean pre-event AUC": f"{auc_pre_mean:.3f}",
                "SEM pre-event AUC": f"{auc_pre_sem:.3f}",

                "post-event AUC window": str(auc_post_window),
                "mean post-event AUC": f"{auc_post_mean:.3f}",
                "SEM post-event AUC": f"{auc_post_sem:.3f}"
            }

            events_info = (
                pd.DataFrame.from_dict(event_parameters, orient="index")
                .reset_index()
            )
            events_info.columns = ["perievent analysis", label_base]
            events_info = events_info.astype(str)

            perievent_results[data_key][event] = {
                "epocs": epocs,
                "epocs_baselined": epocs_baselined,
                "epocs_mean_std_sem": epocs_mean_std_sem,
                "mean_trace": mean_epoc_stream,
                "std_trace": std_epoc_stream,
                "sem_trace": sem_epoc_stream,
                "events_info": events_info,
                "fig_5": fig_5,
                "fig_15": fig_15,
                "subset_figs": subset_figs,
                "event_parameters": event_parameters,
                "n_events": event_counts[event]
            }
            
    for data_key in ["GCaMP", "isosbestic"]:
        for event_name, result_dict in perievent_results[data_key].items():
            print(f"{data_key} | Event: {event_name}")

            display(result_dict["fig_5"])
            display(result_dict["fig_15"])
            plt.close(result_dict["fig_5"])
            plt.close(result_dict["fig_15"])

            for fig in (result_dict.get("subset_figs") or []):
                display(fig)
                plt.close(fig)


        
    
    # ============================
    # DONE
    # ============================
    

        
    def print_structure_with_types(d, indent=0):
        for key, val in d.items():
            prefix = "  " * indent
            if isinstance(val, dict):
                print(f"{prefix}- {key}/")
                print_structure_with_types(val, indent + 1)
            else:
                print(f"{prefix}- {key}: {type(val).__name__}")

    print_structure_with_types(perievent_results)

    if segment_mode == 'during':

        epocs_missed_gcamp = perievent_results["GCaMP"]["drug infusion"]["epocs"]
        display(epocs_missed_gcamp)

        events_info = perievent_results["GCaMP"]["drug infusion"]["events_info"]
        display(events_info)

    # ============================
    # SAVING
    # ============================
    if save:
        
        '''
        feather_folder/
        │
        ├─ {session_id}_{rat}_preproc_info.feather
        ├─ {session_id}_{rat}_preproc_info.txt
        │
        ├─ Peak Detection (if segment_mode != 'transitions')
        │   ├─ {base_path}_peaks_info.feather          ← peak_df
        │   ├─ {base_path}_peak_amps.feather          ← peak amplitudes
        │   ├─ {base_path}_peak_aucs.feather          ← peak AUCs
        │   ├─ {base_path}_peak_df.feather            ← raw_peak_df
        │   ├─ {base_path}_baselined_peak_df.feather ← baselined_peak_df
        │   └─ {base_path}_peaks_info.txt             ← peak_df text version
        │
        ├─ Perievent Results
        │   ├─ {session_id}_{rat}_{data_key}_{event_name}_epocs.feather
        │   ├─ {session_id}_{rat}_{data_key}_{event_name}_epocs_baselined.feather
        │   ├─ {session_id}_{rat}_{data_key}_{event_name}_epocs_mean_std_sem.feather
        │   ├─ {session_id}_{rat}_{data_key}_{event_name}_events_info.feather
        │   ├─ {session_id}_{rat}_{data_key}_{event_name}_events_info.txt
        │   ├─ {session_id}_{rat}_{data_key}_{event_name}_fig_5.png
        │   ├─ {session_id}_{rat}_{data_key}_{event_name}_fig_15.png
        │   ├─ {session_id}_{rat}_{data_key}_{event_name}_subset_1.png
        │   ├─ {session_id}_{rat}_{data_key}_{event_name}_subset_2.png
        │   └─ {session_id}_{rat}_{data_key}_{event_name}_event_parameters.feather
        │
        └─ Other preprocessed streams
            └─ {base_path}_streams_preproc.feather

        '''


        #################################################################################################################################

        # -----------------------------
        os.makedirs(feather_folder, exist_ok=True)
        print(f"\n📁 Files will be saved in:\n   {feather_folder}\n")

        # -----------------------------
        # --- Helper functions ---
        # -----------------------------
        def clean_filename_component(s):
            """Clean strings for use in filenames."""
            return re.sub(r'[<>:"/\\|?*]', '', str(s).replace(' ', '_'))

        def safe_feather_save(df, filepath):
            """Save a DataFrame as Feather with overwrite protection."""
            if os.path.exists(filepath):
                print(f"\n⚠️  File already exists at:\n   {filepath}\n🛑 Skipping save.\n")
                return
            if df is None:
                print(f"\n⚠️  Warning: No DataFrame provided for:\n   {filepath}\n🛑 Skipping save.\n")
                return
            if df.empty:
                print(f"\n⚠️  Warning: Empty DataFrame at:\n   {filepath}\n🛑 Skipping save.\n")
                return

            df = df.reset_index(drop=True)
            df.columns = [str(col) for col in df.columns]
            df.to_feather(filepath)
            print(f"💾 Saved Feather file:\n   {filepath}")

        # -----------------------------
        # --- Base filename ---
        # -----------------------------
        if segment_mode == 'during':
            avail_tag = 'Avail_'
        elif segment_mode == 'before':
            avail_tag = 'Unavail_late_'
        elif segment_mode == 'after':
            avail_tag = 'Unavail_early_'
        elif segment_mode == 'transitions':
            avail_tag = 'transitions_'
        elif segment_mode is None:
            avail_tag = ''
        else:
            raise ValueError(f"❌ Unknown segment_mode: {segment_mode}")

        session_clean = clean_filename_component(session_id)
        rat_clean = clean_filename_component(rat)
        base_path = os.path.join(feather_folder, f"{session_clean}_{avail_tag}{rat_clean}")
        print(f"📦 Base path for this session: {base_path}\n")

        # -----------------------------
        # --- Preprocessing info ---
        # -----------------------------
        if segment_mode != 'transitions':
            preproc_info = {
                "session ID": session_id,
                "rats and box assignments in this group": json.dumps(box_to_rat, indent=4),
                "rat analyzed": rat,
                "new fs after downsample": new_fs,
                "lowpass filter (Hz)": filt,
                #"light artifacts corrected for:": correct_stream,
                "lambda value used for airPLS correction": lambda_,
                "code used for dropsec during stream trimming": dropsec_code,
                "startcode (for selecting time segment)": startcode,
                "time interval dropped from beginning of stream (sec)": (
                    dropsec_all[startcode] if isinstance(dropsec_all, (list, np.ndarray)) and len(dropsec_all) > startcode
                    else "N/A"
                ),
                "duration of examined stream data (min)": keepmin_streams,
            }

            # Feather
            preproc_df_path = base_path + "_preproc_info.feather"
            preproc_info_df = pd.DataFrame.from_dict(preproc_info, orient='index').reset_index()
            preproc_info_df.columns = ['preprocessing info', rat_clean]
            preproc_info_df = preproc_info_df.astype(str)
            safe_feather_save(preproc_info_df, preproc_df_path)

            # TXT
            preproc_txt_path = base_path + "_preproc_info.txt"
            if not os.path.exists(preproc_txt_path):
                with open(preproc_txt_path, 'w') as f:
                    for key, value in preproc_info.items():
                        f.write(f"{key:<45}: {value}\n")
                print(f"📝 Saved preprocessing info (text): {preproc_txt_path}")

        # -----------------------------
        # --- Peak detection ---
        # -----------------------------
        if segment_mode != 'transitions':
            
            if neg_peaks:
                peak_saves = {
                    'peak_df': '_neg_peaks_info',
                    'peak_amps': '_neg_peak_amps',
                    'peak_aucs': '_neg_peak_aucs',
                    'raw_peak_df': '_neg_peak_df',
                    'baselined_peak_df': '_neg_baselined_peak_df'
                }
                
            else:
                peak_saves = {
                    'peak_df': '_pos_peaks_info',
                    'peak_amps': '_pos_peak_amps',
                    'peak_aucs': '_pos_peak_aucs',
                    'raw_peak_df': '_pos_peak_df',
                    'baselined_peak_df': '_pos_baselined_peak_df'
                }
            
            '''
            for var_name, suffix in peak_saves.items():
                if var_name in locals() or var_name in globals():
                    data = locals().get(var_name, globals().get(var_name))
                    if data is not None:
                        safe_feather_save(data, base_path + suffix + ".feather")
                        if var_name == 'peak_df':
                            txt_path = base_path + suffix + ".txt"
                            if not os.path.exists(txt_path):
                                with open(txt_path, 'w') as f:
                                    f.write(data.to_string(index=False))
                                print(f"📝 Saved peak info (text): {txt_path}")
                else:
                    print(f"⚠️ Warning: '{var_name}' not defined. Skipping save.")
            '''
            
            # --- Save each dataframe ---
            for var_name, suffix in peak_saves.items():
                # Get variable from locals/globals
                data = locals().get(var_name, globals().get(var_name))
                if data is not None:
                    # Negate numeric dataframes only
                    if neg_peaks and var_name in ['raw_peak_df', 'baselined_peak_df']:
                        data = -data

                    safe_feather_save(data, base_path + suffix + ".feather")

                    # Save TXT only for peak_df (summary/info)
                    if var_name == 'peak_df':
                        txt_path = base_path + suffix + ".txt"
                        if not os.path.exists(txt_path):
                            with open(txt_path, 'w') as f:
                                f.write(data.to_string(index=False))
                            print(f"📝 Saved peak info (text): {txt_path}")
                else:
                    print(f"⚠️ Warning: '{var_name}' not defined. Skipping save.")


        # -----------------------------
        # --- Perievent results ---
        # -----------------------------
        for data_key, events_dict in perievent_results.items():
            data_safe = clean_filename_component(data_key)

            for event_name, result_dict in events_dict.items():
                event_safe = clean_filename_component(event_name)

                # Save DataFrames
                for df_key, suffix in [
                    ("epocs", "_epocs.feather"),
                    ("epocs_baselined", "_epocs_baselined.feather"),
                    ("epocs_mean_std_sem", "_epocs_mean_std_sem.feather"),
                    ("events_info", "_events_info.feather")
                ]:
                    if df_key in result_dict:
                        safe_feather_save(result_dict[df_key],
                                          os.path.join(feather_folder, f"{session_clean}_{avail_tag}{rat_clean}_{data_safe}_{event_safe}{suffix}"))

                # Save events_info TXT
                if "events_info" in result_dict:
                    txt_path = os.path.join(feather_folder, f"{session_clean}_{avail_tag}{rat_clean}_{data_safe}_{event_safe}_events_info.txt")
                    if not os.path.exists(txt_path):
                        with open(txt_path, "w") as f:
                            f.write(result_dict["events_info"].to_string(index=False))
                        print(f"📝 Saved events info (text): {txt_path}")


                # Save event_parameters
                if "event_parameters" in result_dict:
                    ep_df = pd.DataFrame.from_dict(result_dict["event_parameters"], orient="index").reset_index()
                    ep_df.columns = ["parameter", "value"]
                    ep_df = ep_df.astype(str)  # <-- convert all to string
                    safe_feather_save(
                        ep_df,
                        os.path.join(
                            feather_folder,
                            f"{session_clean}_{avail_tag}{rat_clean}_{data_safe}_{event_safe}_event_parameters.feather"
                        )
                    )



# functions for plotting the raw traces


## stream peek for four boxes, select rat for further analysis


# detect and remove artifacts in the stream data

# filter and downsample the stream data 

## visualize the event TTLs, trimming and filtering streams

## filter the epoc_data to examine the events occurring during the photometry time segment of interest

## remove light artifacts if necessary



## motion correction using a linear fit between isosbestic and GCaMP streams

## Correct photobleaching for isosbestic and 465 channels by subtracting a double exponential fit 

## Correct for photobleaching, baseline drift using an adaptive iteratively reweighted penalized least squares algorithm (airPLS)


## normalization by Z-scoring (subtract the mean, divide by the standard deviation) and dF/F (subtract the motion correction, then divide by the estimated motion from above)

# peak detection

# function for extracting and averaging data around epoc events 

##   to load previously saved dataframes of epoc and stream data

## epoc stream extraction and averaging

### an alternate function that compares the peri-event data with random 

# save the pre-processed streams and peri-event data as feather files
